In [11]:
import json
import asyncio
import re
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse, parse_qs

from playwright.async_api import async_playwright, TimeoutError as PWTimeoutError
from bs4 import BeautifulSoup


# =========================
# Config (피해구제 사례 115)
# =========================
BASE = "https://www.consumer.go.kr"

LIST_URL_TMPL = (
    "https://www.consumer.go.kr/user/ftc/consumer/dmgerlifcase/115/selectDmgeRlifCaseList.do"
    "?page={page}&row=25&searchType=&searchCnd=&searchWrd="
)

OUT_JSONL = "dmge_rlif_cases_115_full.jsonl"
ERROR_JSONL = "dmge_rlif_cases_115_errors.jsonl"

HEADLESS = True
TIMEOUT_MS = 30_000

MAX_RETRIES = 3
RETRY_BACKOFF_SEC = 1.5
CHECKPOINT_EVERY = 1

# selectors (네 HTML 기준)
LIST_ROW_SELECTOR = "table.tbl.col.data tbody tr"
DETAIL_TABLE_SELECTOR = "table.tbl.row.data"

DETAIL_CONCURRENCY = 3


# =========================
# Utils
# =========================
def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def extract_case_sn(url: str) -> str | None:
    qs = parse_qs(urlparse(url).query)
    v = qs.get("dmgeRlifCaseSn")
    return v[0] if v else None

def make_doc_id(url: str) -> str:
    sn = extract_case_sn(url)
    return str(sn) if sn else f"url:{url}"

def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

def safe_text(el) -> str:
    if not el:
        return ""
    return normalize_text(el.get_text("\n", strip=True))


# =========================
# Parsing: 상세 (115)
# =========================
def parse_detail_html(html: str, url: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")

    def get_row(label: str) -> str:
        for th in soup.find_all("th"):
            if label in th.get_text(strip=True):
                td = th.find_next_sibling("td")
                if not td:
                    return ""
                div = td.select_one("div.bbs_view_content")
                return safe_text(div) if div else safe_text(td)
        return ""

    title = get_row("제목")
    category = get_row("분류")
    source = get_row("출처")
    views_detail = get_row("조회수")   # ✅ 115는 상세에도 조회수 존재
    question = get_row("질문")
    answer = get_row("답변")

    parts = []
    if title: parts.append(f"제목: {title}")
    if category: parts.append(f"분류: {category}")
    if source: parts.append(f"출처: {source}")
    if views_detail: parts.append(f"조회수: {views_detail}")
    if question: parts.append(f"질문:\n{question}")
    if answer: parts.append(f"답변:\n{answer}")

    content = "\n\n".join(parts).strip()

    return {
        "id": make_doc_id(url),
        "url": url,
        "title": title,
        "source": source,
        "category": category,
        "views_detail": views_detail,
        "question": question,
        "answer": answer,
        "content": content,
        "collected_at": now_iso(),
        "metadata": {
            "site": "consumer.go.kr",
            "doc_type": "dmge_rlif_case",
            "dmgeRlifCaseSn": extract_case_sn(url),
        },
    }


# =========================
# IO
# =========================
def load_seen_ids(path: str) -> set[str]:
    seen = set()
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    _id = obj.get("id")
                    if _id:
                        seen.add(str(_id))
                except:
                    pass
    except FileNotFoundError:
        pass
    return seen

def append_jsonl(fp, obj: dict):
    fp.write(json.dumps(obj, ensure_ascii=False) + "\n")
    fp.flush()


# =========================
# Network optimization
# =========================
async def block_heavy_assets(route):
    rtype = route.request.resource_type
    if rtype in ("image", "font", "media", "stylesheet"):
        await route.abort()
    else:
        await route.continue_()


# =========================
# 목록 메타 추출 (115 구조)
# - 번호 / 제목(link) / 출처 / 조회수
# =========================
async def extract_list_items(list_page) -> list[dict]:
    items = await list_page.locator(LIST_ROW_SELECTOR).evaluate_all(
        """rows => rows.map(tr => {
            const get = (sel) => {
                const el = tr.querySelector(sel);
                return el ? el.textContent.trim() : "";
            };

            const a = tr.querySelector("td.title a");
            const href = a ? a.getAttribute("href") : "";
            const title_list = a ? a.textContent.trim() : "";

            return {
                href,
                title_list,
                no: get('td[aria-label="번호"]'),
                source_list: get('td[aria-label="출처"]'),
                views_list: get('td[aria-label="조회수"]'),
            };
        })"""
    )

    cleaned = []
    for it in items:
        href = it.get("href") or ""
        if not href:
            continue
        it["url"] = urljoin(BASE, href)
        it["case_sn"] = extract_case_sn(it["url"])
        cleaned.append(it)
    return cleaned


# =========================
# Main
# =========================
async def main(start_page=1, end_page=50):
    seen = load_seen_ids(OUT_JSONL)
    print("seen already:", len(seen))

    sem = asyncio.Semaphore(DETAIL_CONCURRENCY)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)

        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
            )
        )
        context.set_default_timeout(TIMEOUT_MS)
        await context.route("**/*", block_heavy_assets)

        list_page = await context.new_page()

        saved = 0
        skipped = 0
        failed = 0

        with open(OUT_JSONL, "a", encoding="utf-8") as out_fp, \
             open(ERROR_JSONL, "a", encoding="utf-8") as err_fp:

            async def fetch_detail(url: str, page_no: int, list_meta: dict):
                nonlocal saved, skipped, failed

                async with sem:
                    doc_id = make_doc_id(url)
                    if doc_id in seen:
                        skipped += 1
                        return

                    detail_page = await context.new_page()
                    try:
                        last_err = None
                        for attempt in range(1, MAX_RETRIES + 1):
                            try:
                                await detail_page.goto(url, wait_until="domcontentloaded")
                                await detail_page.wait_for_selector(DETAIL_TABLE_SELECTOR)

                                html = await detail_page.content()
                                item = parse_detail_html(html, url)

                                if not (item["title"] or item["question"] or item["answer"]):
                                    raise RuntimeError("empty parsed fields")

                                # ✅ 목록 메타 병합
                                item["list_meta"] = {
                                    "list_page": page_no,
                                    "no": list_meta.get("no", ""),
                                    "title_list": list_meta.get("title_list", ""),
                                    "source_list": list_meta.get("source_list", ""),
                                    "views_list": list_meta.get("views_list", ""),
                                }

                                # ✅ metadata 필터용
                                item["metadata"].update({
                                    "no": list_meta.get("no", ""),
                                    "source_list": list_meta.get("source_list", ""),
                                    "views_list": list_meta.get("views_list", ""),
                                })

                                append_jsonl(out_fp, item)
                                seen.add(item["id"])
                                saved += 1
                                return

                            except Exception as e:
                                last_err = e
                                if attempt < MAX_RETRIES:
                                    await asyncio.sleep(RETRY_BACKOFF_SEC * attempt)
                                else:
                                    failed += 1
                                    append_jsonl(err_fp, {
                                        "url": url,
                                        "id": doc_id,
                                        "page": page_no,
                                        "error": repr(last_err),
                                        "at": now_iso(),
                                    })
                    finally:
                        await detail_page.close()

            for pg in range(start_page, end_page + 1):
                list_url = LIST_URL_TMPL.format(page=pg)

                list_items = []
                for attempt in range(1, MAX_RETRIES + 1):
                    try:
                        await list_page.goto(list_url, wait_until="domcontentloaded")
                        await list_page.wait_for_selector(LIST_ROW_SELECTOR)
                        list_items = await extract_list_items(list_page)

                        if len(list_items) == 0:
                            raise RuntimeError("list empty (0 rows with href)")

                        break

                    except (PWTimeoutError, Exception) as e:
                        if attempt < MAX_RETRIES:
                            await asyncio.sleep(RETRY_BACKOFF_SEC * attempt)
                        else:
                            tr_cnt = await list_page.locator("tbody tr").count()
                            print(f"[list][fail] page={pg} tr={tr_cnt} err={repr(e)}")
                            append_jsonl(err_fp, {
                                "url": list_url,
                                "page": pg,
                                "error": f"list_page_failed: {repr(e)}",
                                "at": now_iso(),
                            })
                            list_items = []

                if not list_items:
                    continue

                print(f"[list] page={pg} items={len(list_items)}")

                tasks = [fetch_detail(meta["url"], pg, meta) for meta in list_items]
                await asyncio.gather(*tasks)

                if pg % CHECKPOINT_EVERY == 0:
                    print(f"✅ checkpoint page {pg} | saved={saved} skipped={skipped} failed={failed}")

        await browser.close()

    print("\n==== DONE ====")
    print(f"pages: {start_page}~{end_page}")
    print(f"saved: {saved}")
    print(f"skipped(seen): {skipped}")
    print(f"failed: {failed}")


# 실행 예시
await main(start_page=1, end_page=60)   # 1497건이면 대략 60페이지 정도
# await main(start_page=1, end_page=10)


seen already: 0
[list] page=1 items=25
✅ checkpoint page 1 | saved=25 skipped=0 failed=0
[list] page=2 items=25
✅ checkpoint page 2 | saved=50 skipped=0 failed=0
[list] page=3 items=25
✅ checkpoint page 3 | saved=75 skipped=0 failed=0
[list] page=4 items=25
✅ checkpoint page 4 | saved=100 skipped=0 failed=0
[list] page=5 items=25
✅ checkpoint page 5 | saved=125 skipped=0 failed=0
[list] page=6 items=25
✅ checkpoint page 6 | saved=150 skipped=0 failed=0
[list] page=7 items=25
✅ checkpoint page 7 | saved=175 skipped=0 failed=0
[list] page=8 items=25
✅ checkpoint page 8 | saved=200 skipped=0 failed=0
[list] page=9 items=25
✅ checkpoint page 9 | saved=225 skipped=0 failed=0
[list] page=10 items=25
✅ checkpoint page 10 | saved=250 skipped=0 failed=0
[list] page=11 items=25
✅ checkpoint page 11 | saved=275 skipped=0 failed=0
[list] page=12 items=25
✅ checkpoint page 12 | saved=300 skipped=0 failed=0
[list] page=13 items=25
✅ checkpoint page 13 | saved=325 skipped=0 failed=0
[list] page=14 it

In [ ]:
import json

INPUT = "dmgerlif_cases_full.jsonl"
OUTPUT = "dmgerlif_cases_full.json"

data = []
with open(INPUT, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"✅ 변환 완료: {OUTPUT} (총 {len(data)}건)")
